# Download LiveCodeBench release_v6

This CPU-only notebook downloads the nine pinned Parquet shards used by `FedAvg_mopsd.ipynb`. It downloads one shard at a time, retries interrupted HTTP transfers with Range requests, prints plain-text progress directly in Jupyter, reuses completed Hub-cache files, and does not load a model or use a GPU.

In [ ]:
# Papermill parameters
LOCAL_DIR = "/root/datasets/livecodebench_code_generation_lite"
HF_ENDPOINT = "https://hf-mirror.com"
MAX_RETRIES = 100
CONNECT_TIMEOUT = 60
READ_TIMEOUT = 60
PROGRESS_INTERVAL_SECONDS = 5
CHUNK_SIZE_MB = 1


In [ ]:
import os

# This notebook never imports torch or loads a model. Hide all GPUs explicitly.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["HF_ENDPOINT"] = HF_ENDPOINT.rstrip("/")
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = str(READ_TIMEOUT)
os.environ["HF_HUB_ETAG_TIMEOUT"] = str(CONNECT_TIMEOUT)
os.environ.pop("HF_HUB_DISABLE_PROGRESS_BARS", None)

print(f"HF endpoint: {os.environ['HF_ENDPOINT']}", flush=True)
print("CUDA_VISIBLE_DEVICES is empty; this download uses CPU and network only.", flush=True)


In [ ]:
from pathlib import Path
from urllib.parse import quote
import shutil
import time

import requests
from huggingface_hub import HfApi

REPO_ID = "livecodebench/code_generation_lite"
REVISION = "d44be6b144381afa49392b1f0eb424a64a4d8a10"
RELEASE = "release_v6"
SHARD_COUNT = 9

local_root = Path(LOCAL_DIR).expanduser().resolve()
release_dir = local_root / RELEASE
release_dir.mkdir(parents=True, exist_ok=True)

print(f"Repository:  {REPO_ID}", flush=True)
print(f"Revision:    {REVISION}", flush=True)
print(f"Destination: {release_dir}", flush=True)
print("Mode:        sequential HTTP Range download with automatic resume", flush=True)

api = HfApi(endpoint=os.environ["HF_ENDPOINT"])
repo_info = api.dataset_info(
    repo_id=REPO_ID,
    revision=REVISION,
    files_metadata=True,
)
remote_metadata = {item.rfilename: item for item in repo_info.siblings}
remote_paths = [
    f"{RELEASE}/test-{index:05d}-of-00009.parquet"
    for index in range(SHARD_COUNT)
]
expected_sizes = {}
for remote_path in remote_paths:
    item = remote_metadata.get(remote_path)
    if item is None or item.size is None:
        raise RuntimeError(f"Could not obtain the expected size of {remote_path}")
    expected_sizes[remote_path] = int(item.size)

# Reuse complete shards from an earlier load_dataset()/Hub-cache download.
hub_cache = Path(
    os.environ.get("HF_HUB_CACHE", "~/.cache/huggingface/hub")
).expanduser().resolve()
cached_release = (
    hub_cache
    / "datasets--livecodebench--code_generation_lite"
    / "snapshots"
    / REVISION
    / RELEASE
)
reused = 0
if cached_release.is_dir():
    for remote_path in remote_paths:
        filename = Path(remote_path).name
        source = cached_release / filename
        destination = release_dir / filename
        expected_size = expected_sizes[remote_path]
        if (
            source.is_file()
            and source.stat().st_size == expected_size
            and not destination.exists()
        ):
            try:
                os.link(source.resolve(), destination)
                method = "hardlink"
            except OSError:
                shutil.copy2(source.resolve(), destination)
                method = "copy"
            reused += 1
            print(f"[REUSE:{method}] {filename}", flush=True)
print(f"Reused complete Hub-cache shards: {reused}/{SHARD_COUNT}", flush=True)

def format_seconds(seconds):
    if seconds is None or seconds == float("inf"):
        return "--:--:--"
    seconds = max(0, int(seconds))
    return f"{seconds // 3600:02d}:{(seconds % 3600) // 60:02d}:{seconds % 60:02d}"

def print_progress(filename, current, total, started_at, resumed_from):
    elapsed = max(time.monotonic() - started_at, 1e-6)
    transferred = max(current - resumed_from, 0)
    speed = transferred / elapsed
    eta = (total - current) / speed if speed > 0 else None
    percent = 100.0 * current / total
    print(
        f"[PROGRESS] {filename}: {current / 2**20:.2f}/{total / 2**20:.2f} MiB "
        f"({percent:.2f}%) speed={speed / 2**20:.2f} MiB/s "
        f"elapsed={format_seconds(elapsed)} eta={format_seconds(eta)}",
        flush=True,
    )

def download_one(remote_path, destination, expected_size):
    part_path = destination.with_name(destination.name + ".part")

    if destination.is_file():
        actual_size = destination.stat().st_size
        if actual_size == expected_size:
            print(f"[SKIP] {destination.name}: complete local file", flush=True)
            return
        if not part_path.exists():
            destination.replace(part_path)
        else:
            destination.unlink()
        print(
            f"[REPAIR] {destination.name}: local size {actual_size} does not match "
            f"expected size {expected_size}",
            flush=True,
        )

    if part_path.exists() and part_path.stat().st_size > expected_size:
        part_path.unlink()

    resolve_url = (
        f"{os.environ['HF_ENDPOINT']}/datasets/{REPO_ID}/resolve/"
        f"{REVISION}/{quote(remote_path, safe='/')}?download=true"
    )
    session = requests.Session()
    session.headers.update({"Accept-Encoding": "identity"})

    for attempt in range(1, int(MAX_RETRIES) + 1):
        offset = part_path.stat().st_size if part_path.exists() else 0
        if offset == expected_size:
            os.replace(part_path, destination)
            print(f"[DONE] {destination.name}: {expected_size / 2**20:.2f} MiB", flush=True)
            return

        headers = {"Range": f"bytes={offset}-"} if offset else {}
        print(
            f"[CONNECT] {destination.name}: attempt={attempt}/{MAX_RETRIES}, "
            f"resume_from={offset / 2**20:.2f} MiB",
            flush=True,
        )
        try:
            with session.get(
                resolve_url,
                headers=headers,
                stream=True,
                allow_redirects=True,
                timeout=(int(CONNECT_TIMEOUT), int(READ_TIMEOUT)),
            ) as response:
                if response.status_code == 416 and offset == expected_size:
                    os.replace(part_path, destination)
                    print(f"[DONE] {destination.name}", flush=True)
                    return
                response.raise_for_status()

                if offset and response.status_code != 206:
                    print(
                        f"[RESTART] {destination.name}: server ignored HTTP Range",
                        flush=True,
                    )
                    offset = 0
                    mode = "wb"
                else:
                    mode = "ab" if offset else "wb"

                started_at = time.monotonic()
                resumed_from = offset
                last_report = started_at - float(PROGRESS_INTERVAL_SECONDS)
                current = offset
                with part_path.open(mode) as output_file:
                    for chunk in response.iter_content(chunk_size=int(CHUNK_SIZE_MB) * 2**20):
                        if not chunk:
                            continue
                        output_file.write(chunk)
                        current += len(chunk)
                        now = time.monotonic()
                        if now - last_report >= float(PROGRESS_INTERVAL_SECONDS):
                            print_progress(
                                destination.name, current, expected_size, started_at, resumed_from
                            )
                            last_report = now

                actual_size = part_path.stat().st_size
                print_progress(
                    destination.name, actual_size, expected_size, started_at, resumed_from
                )
                if actual_size == expected_size:
                    os.replace(part_path, destination)
                    print(f"[DONE] {destination.name}: {actual_size / 2**20:.2f} MiB", flush=True)
                    return
                if actual_size > expected_size:
                    part_path.unlink()
                    raise RuntimeError(
                        f"downloaded size {actual_size} exceeds expected size {expected_size}"
                    )
                raise IOError(
                    f"connection ended at {actual_size}/{expected_size} bytes"
                )
        except (requests.RequestException, OSError, RuntimeError) as exc:
            current = part_path.stat().st_size if part_path.exists() else 0
            if attempt >= int(MAX_RETRIES):
                raise RuntimeError(
                    f"Failed to download {remote_path} after {MAX_RETRIES} attempts; "
                    f"partial file retained at {part_path} ({current} bytes)"
                ) from exc
            delay = min(2 ** min(attempt - 1, 5), 30)
            print(
                f"[RETRY] {destination.name}: {type(exc).__name__}: {exc}; "
                f"saved={current / 2**20:.2f} MiB, retrying in {delay}s",
                flush=True,
            )
            time.sleep(delay)

missing_remote_files = []
for remote_path in remote_paths:
    destination = release_dir / Path(remote_path).name
    if not (destination.is_file() and destination.stat().st_size == expected_sizes[remote_path]):
        missing_remote_files.append(remote_path)

if missing_remote_files:
    print(f"Downloading missing shards sequentially: {len(missing_remote_files)}/{SHARD_COUNT}", flush=True)
    for position, remote_path in enumerate(missing_remote_files, start=1):
        print(
            f"\n[SHARD] {position}/{len(missing_remote_files)}: {Path(remote_path).name}",
            flush=True,
        )
        download_one(
            remote_path,
            release_dir / Path(remote_path).name,
            expected_sizes[remote_path],
        )
else:
    print("All nine shards are already available locally.", flush=True)


In [ ]:
files = [
    release_dir / f"test-{index:05d}-of-00009.parquet"
    for index in range(SHARD_COUNT)
]
invalid = []
for index, path in enumerate(files):
    remote_path = f"{RELEASE}/test-{index:05d}-of-00009.parquet"
    expected_size = expected_sizes[remote_path]
    actual_size = path.stat().st_size if path.is_file() else None
    if actual_size != expected_size:
        invalid.append((path, actual_size, expected_size))
if invalid:
    raise RuntimeError(
        "Download ended with missing or size-mismatched shards: " + repr(invalid)
    )

total_bytes = sum(path.stat().st_size for path in files)
print("\n[COMPLETE] LiveCodeBench local dataset is ready.", flush=True)
for path in files:
    print(f"{path.stat().st_size / 2**20:9.2f} MiB  {path}", flush=True)
print(f"Total: {total_bytes / 2**30:.3f} GiB", flush=True)
print(f"FedAvg_mopsd.ipynb LCB_DATA_DIR = {str(release_dir)!r}", flush=True)
